# Fine-tune the Preflight embedder on a Colab GPU

Trains `BAAI/bge-large-en-v1.5` on ASRS synopsis→narrative pairs (`preflight corpus finetune`). The pairs come from the local corpus database, with every report the evals read held out. Colab can't reach that database, so export them first:

```bash
uv run preflight corpus finetune --export-pairs data/finetune/asrs_pairs.jsonl
```

1. **Runtime → Change runtime type → T4 GPU.**
2. Upload `asrs_pairs.jsonl` to Google Drive at `MyDrive/preflight-ft/asrs_pairs.jsonl`.
3. Run all cells. Checkpoints are written to Drive every 25 steps. If Colab disconnects, reconnect and run all cells again; training resumes from the last checkpoint.
4. Download `MyDrive/preflight-ft/bge-large-asrs` (without `checkpoints/`) to `models/bge-large-asrs` in the repo, then run `uv run preflight eval embedder --models BAAI/bge-large-en-v1.5 models/bge-large-asrs`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FT_DIR = '/content/drive/MyDrive/preflight-ft'
PAIRS = f'{FT_DIR}/asrs_pairs.jsonl'
OUT = f'{FT_DIR}/bge-large-asrs'
import os
assert os.path.exists(PAIRS), f'upload asrs_pairs.jsonl to {PAIRS} first'
!wc -l "{PAIRS}"

In [ ]:
BRANCH = 'accuracy/embedder-finetune'
!rm -rf /content/preflight && git clone -q --depth 1 -b {BRANCH} https://github.com/BhavyaPatel19/preflight.git /content/preflight
!pip install -q -e /content/preflight "sentence-transformers>=6.0" "datasets>=3.0"
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
%cd /content/preflight
# --resume is safe on a fresh start: it only resumes if OUT/checkpoints already has one.
!preflight corpus finetune --pairs-file "{PAIRS}" --base BAAI/bge-large-en-v1.5 \
    --out "{OUT}" --batch-size 128 --mini-batch-size 16 --resume

In [ ]:
# Final model files to download (checkpoints/ is not needed once training is done)
!ls -la "{OUT}"
!cat "{OUT}/preflight_finetune.json"